# RO-077 Physics simulators: every number in the Note

The tiny simulator lives in `images/simkit.py`: a 1 kg box falls from 0.1 m with a sideways speed of 2.0 m/s onto a
floor with a coefficient of friction of 0.4. Each step finds the contact, adds up the forces, and advances with
semi-implicit Euler, in MuJoCo's order. Nothing is random, so there is no seed.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import simkit as s
r3, r4 = (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))
print('box', s.M, 'kg, drop', s.H0, 'm, sideways', s.VX0, 'm/s, friction', s.MU, '; contact time constant', s.TAU,
      's -> stiffness', s.K, 'N/m, damping', s.B, 'N s/m; sinks', r4(s.M * s.G / s.K), 'm at rest')

box 1.0 kg, drop 0.1 m, sideways 2.0 m/s, friction 0.4 ; contact time constant 0.02 s -> stiffness 2500.0 N/m, damping 100.0 N s/m; sinks 0.0039 m at rest


## 1. The exact answer (hard floor, by hand)

In [2]:
ref = s.reference()
for k, v in ref.items():
    print(k, r4(v))
print('impact: normal impulse', r3(s.M * ref['vz_land']), 'N s; friction impulse up to', r3(s.MU * s.M * ref['vz_land']), 'N s')

t_land 0.1428
vz_land 1.4007
x_land 0.2856
vx_after 1.4397
t_slide 0.3669
d_slide 0.2641
x_stop 0.5497
t_stop 0.5097
impact: normal impulse 1.401 N s; friction impulse up to 0.56 N s


## 2. One step, two Euler rules (free fall, coarse step 0.025 s)

In [3]:
dt = 0.025
semi, expl = s.free_fall(dt, 6, 'semi'), s.free_fall(dt, 6, 'explicit')
exact = [s.H0 - 0.5 * s.G * (k * dt) ** 2 for k in range(7)]
print('semi-implicit', np.round(semi[:, 1], 4).tolist())
print('explicit     ', np.round(expl[:, 1], 4).tolist())
print('exact        ', np.round(exact, 4).tolist())
print('step 1: v =', r3(semi[1, 2]), 'semi z =', r4(semi[1, 1]), 'explicit z =', r4(expl[1, 1]), 'exact z =', r4(exact[1]))
print('exact landing', r4(ref['t_land']), 's; semi-implicit first inside the floor at step', int(np.argmax(semi[:, 1] < 0)),
      '(', r3(semi[int(np.argmax(semi[:, 1] < 0)), 0]), 's)')

semi-implicit [0.1, 0.0939, 0.0816, 0.0632, 0.0387, 0.008, -0.0288]
explicit      [0.1, 0.1, 0.0939, 0.0816, 0.0632, 0.0387, 0.008]
exact         [0.1, 0.0969, 0.0877, 0.0724, 0.0509, 0.0234, -0.0104]
step 1: v = -0.245 semi z = 0.0939 explicit z = 0.1 exact z = 0.0969
exact landing 0.1428 s; semi-implicit first inside the floor at step 6 ( 0.15 s)


## 3. The running example at three time steps (Figure 2)

In [4]:
for dt in (0.0005,) + s.DTS:
    r = s.simulate(dt, T=1.0); t, x, z, vx, vz, N, f, d = r.T
    after = t > ref['t_land'] + 0.05
    stop = np.where((vx == 0) & (t > 0.1))[0]
    print(f"dt {dt}: {len(t) - 1} steps; ends at x {r3(x[-1])} m; deepest {r4(d.max())} m; highest after landing {r3(z[after].max())} m;"
          f" stops sliding at {t[stop[0]] if len(stop) else None} s")
r4s = s.simulate(0.03, T=4.0); t, x, z = r4s[:, 0], r4s[:, 1], r4s[:, 2]
peaks = [(r3(t[i]), r3(z[i])) for i in range(1, len(z) - 1) if z[i] > z[i - 1] and z[i] >= z[i + 1] and z[i] > 0.001]
print('dt 0.03 bounce tops (s, m):', peaks)

dt 0.0005: 2000 steps; ends at x 0.551 m; deepest 0.0114 m; highest after landing -0.004 m; stops sliding at 0.51 s
dt 0.002: 500 steps; ends at x 0.549 m; deepest 0.0097 m; highest after landing -0.004 m; stops sliding at 0.51 s
dt 0.015: 67 steps; ends at x 0.528 m; deepest 0.0214 m; highest after landing 0.088 m; stops sliding at 0.585 s
dt 0.03: 33 steps; ends at x 0.3 m; deepest 0.0324 m; highest after landing 1.361 m; stops sliding at 0.18 s
dt 0.03 bounce tops (s, m): [(0.69, 1.361), (3.09, 16.974)]


## 4. Why a large step blows up: the damper's factor per step

In [5]:
for dt in s.DTS:
    print('dt', dt, ': b dt / m =', r3(s.B * dt / s.M), '; factor 1 - b dt / m =', r3(s.step_factor(dt)))
print('limit: factor below -1 when dt > 2 m / b =', 2 * s.M / s.B, 's')
for dt in s.DTS:
    A = np.array([[1 - s.K * dt * dt / s.M, dt * (1 - s.B * dt / s.M)], [-s.K * dt / s.M, 1 - s.B * dt / s.M]])
    print('dt', dt, ': with the spring too, largest growth per step', r3(max(abs(np.linalg.eigvals(A)))))
print("MuJoCo's rule: time constant at least 2 steps -> largest step", s.TAU / 2, 's for 0.02 s')

dt 0.002 : b dt / m = 0.2 ; factor 1 - b dt / m = 0.8
dt 0.015 : b dt / m = 1.5 ; factor 1 - b dt / m = -0.5
dt 0.03 : b dt / m = 3.0 ; factor 1 - b dt / m = -2.0
limit: factor below -1 when dt > 2 m / b = 0.02 s
dt 0.002 : with the spring too, largest growth per step 0.927
dt 0.015 : with the spring too, largest growth per step 0.739
dt 0.03 : with the spring too, largest growth per step 3.779
MuJoCo's rule: time constant at least 2 steps -> largest step 0.01 s for 0.02 s


## 5. Contact: hard and soft (Figure 3)

In [6]:
H = s.simulate(0.002, T=1.0, contact='hard'); kh = int(np.argmax(H[:, 5] > 0))
print('hard: contact in the step ending at', H[kh, 0], 's; impulse', r3(H[kh, 5] * 0.002), 'N s; friction impulse',
      r3(-H[kh, 6] * 0.002), 'N s; sideways', r3(H[kh - 1, 3]), '->', r3(H[kh, 3]), 'm/s; lowest height', H[:, 2].min(),
      '; stops at x', r3(H[-1, 1]))
S = s.simulate(0.002, T=1.0); i = int(np.argmax(S[:, 7]))
print('soft: deepest', r4(S[i, 7]), 'm at', S[i, 0], 's; largest push', r3(S[:, 5].max()), 'N at', S[int(np.argmax(S[:, 5])), 0],
      's; at rest', r4(S[-1, 7]), 'm deep, push', r3(S[-1, 5]), 'N; stops at x', r3(S[-1, 1]))
d0 = S[i, 7]
print('check at the deepest point (speed about 0): 2500 x', r4(d0), '=', r3(2500 * d0), 'N')
print('at rest: 2500 x', r4(s.M * s.G / s.K), '=', r3(2500 * s.M * s.G / s.K), 'N = weight', s.M * s.G)

hard: contact in the step ending at 0.142 s; impulse 1.393 N s; friction impulse 0.557 N s; sideways 2.0 -> 1.443 m/s; lowest height 0.0 ; stops at x 0.547
soft: deepest 0.0097 m at 0.164 s; largest push 140.046 N at 0.144 s; at rest 0.0039 m deep, push 9.81 N; stops at x 0.549
check at the deepest point (speed about 0): 2500 x 0.0097 = 24.362 N
at rest: 2500 x 0.0039 = 9.81 N = weight 9.81


## 6. Friction holding, then slipping (Figure 4), and the robot's wheel

In [7]:
P = s.push_test(); t, x, z, vx, vz, N, f, d = P.T
ks = int(np.argmax(np.abs(vx) > 0))
print('limit mu m g =', r3(s.MU * s.M * s.G), 'N; cone half-angle', round(math.degrees(math.atan(s.MU)), 1), 'deg')
print('slips in the step ending at', t[ks], 's (push', r3(2 * t[ks - 1]), 'N at its start); at 3.0 s: x', r3(x[-1]), 'm, speed', r3(vx[-1]), 'm/s')
print('after slipping: acceleration (2t - 3.924) / 1; speed at 3 s by hand', r3(((3.0 - 1.962) ** 2)))
a_robot = 0.5
print('robot: 0.5 m/s^2 needs', a_robot, 'N per kg of friction; the floor gives up to', r3(s.MU * s.G), 'N per kg; ratio', r3(a_robot / (s.MU * s.G)))
print('a wheel slips above', r3(s.MU * s.G), 'm/s^2')

limit mu m g = 3.924 N; cone half-angle 21.8 deg
slips in the step ending at 1.966 s (push 3.928 N at its start); at 3.0 s: x 0.373 m, speed 1.075 m/s
after slipping: acceleration (2t - 3.924) / 1; speed at 3 s by hand 1.077
robot: 0.5 m/s^2 needs 0.5 N per kg of friction; the floor gives up to 3.924 N per kg; ratio 0.127
a wheel slips above 3.924 m/s^2
